# Test qwen

In [1]:
%cd /mnt/data1tb/thangcn/datnv2

/mnt/data1tb/thangcn/datnv2


In [2]:
from vllm import LLM, SamplingParams
import json
from openai import OpenAI
from langchain_openai import ChatOpenAI
from service.func_for_fc import rag_service_info, rag_product_info, rag_doctor_info, qa_medical, qa_symptom, book_appointment

/home/duyhoang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 05-23 09:03:32 [__init__.py:239] Automatically detected platform cuda.


2025-05-23 09:03:32,957	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/mnt/data1tb/thangcn/datnv2/service/func_for_fc.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [5]:
from typing import List, Dict, Any
from langchain_core.runnables.history import RunnableWithMessageHistory
from prompts.prompt import contextualize_q_system_prompt, qa_system_prompt
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_openai import ChatOpenAI
from service.func_for_fc import rag_doctor_info, rag_product_info, rag_service_info, book_appointment
import os
import json
import time
import streamlit as st
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
# import uuid
import atexit
from service.message_stored import load_session_history, get_db, save_message

In [6]:
load_dotenv('/mnt/data1tb/thangcn/datnv2/.env')
open_ai_key = os.getenv("OPENAI_API_KEY")
# groq_api_key = os.getenv("GROQ_API_KEY")
EMBED_MODEL = "nampham1106/bkcare-embedding" #os.getenv("EMBED_MODEL", "nampham1106/bkcare-embedding")

store = {}
session_id = 'thangcn1'

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = load_session_history(session_id)
    return store[session_id]

def save_all_sessions():
    for session_id, chat_history in store.items():
        for message in chat_history.messages:
            save_message(session_id, message["role"], message["content"])

atexit.register(save_all_sessions)

with open('/mnt/data1tb/thangcn/datnv2/prompts/tools.json', 'r') as f:
    function_schema = json.load(f)

tools = [
    {
        "type": "function",
        "function": tool
    } for tool in function_schema
]

llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
    temperature=0.8,
    model="thang1943/Qwen2.5-7B-Instruct-final",
)

def create_contextualize_prompt(contextualize_q_system_prompt, qa_system_prompt):
    contextualize_q_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", contextualize_q_system_prompt), 
            MessagesPlaceholder("chat_history"), 
            ("human", "{input}")
        ]
    )

    qa_prompt = ChatPromptTemplate.from_messages(
        [("system", qa_system_prompt), MessagesPlaceholder("chat_history"), ("human", "{input}")]
    )

    return contextualize_q_prompt, qa_prompt


contextualize_q_prompt, qa_prompt = create_contextualize_prompt(contextualize_q_system_prompt, qa_system_prompt)


In [7]:
def process_llm_function_call(user_prompt: str):
    # messages = chat_history.messages
    messages = []
    # Thêm câu hỏi mới nhất
    messages.append(
        {"role": "user", "content": user_prompt}
    ) 
    # Gọi LLM với function calling
    response = llm.predict_messages(
        messages,
        tools=tools,
        tool_choice="auto",
    )
    print(response)
    return response

In [21]:
user_prompt = "Tôi muốn đặt lịch khám tim mạch ngày 2025-09-10 lúc 10:00, tên là Nguyễn Văn B, số điện thoại 0909123456."

In [22]:
r = process_llm_function_call(user_prompt)

content='' additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-127fa6be0e2c476198e61a4d082d754c', 'function': {'arguments': '{"name": "Nguyễn Văn B", "phone": "0909123456", "date": "2025-09-10", "time": "10:00", "specialty": "tim mạch"}', 'name': 'book_appointment'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 800, 'total_tokens': 873, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'thang1943/Qwen2.5-7B-Instruct-final', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None} id='run-f84eba39-1072-49ec-9317-9fcc596299d6-0' tool_calls=[{'name': 'book_appointment', 'args': {'name': 'Nguyễn Văn B', 'phone': '0909123456', 'date': '2025-09-10', 'time': '10:00', 'specialty': 'tim mạch'}, 'id': 'chatcmpl-tool-127fa6be0e2c476198e61a4d082d754c', 'type': 'tool_call'}] usage_metadata={'input_tokens': 800, 'output_tokens': 73, 'total_tokens': 873, 'input_token_details':

In [23]:
x = r.additional_kwargs
x

{'tool_calls': [{'id': 'chatcmpl-tool-127fa6be0e2c476198e61a4d082d754c',
   'function': {'arguments': '{"name": "Nguyễn Văn B", "phone": "0909123456", "date": "2025-09-10", "time": "10:00", "specialty": "tim mạch"}',
    'name': 'book_appointment'},
   'type': 'function'}],
 'refusal': None}

In [25]:
x['tool_calls'][0]['function']['arguments']

'{"name": "Nguyễn Văn B", "phone": "0909123456", "date": "2025-09-10", "time": "10:00", "specialty": "tim mạch"}'

In [27]:
function_args = json.loads(r.additional_kwargs['tool_calls'][0]['function']['arguments'])